In [0]:
!pip install ccxt xgboost

In [0]:
import ccxt
import pandas as pd
import numpy as np
import time
from datetime import datetime
import xgboost as xgb

# API da Binance
exchange = ccxt.binanceus({'enableRateLimit': True})
par_altcoin = 'ONT/USDT'
par_btc = 'BTC/USDT'
timeframe = '1h'

# model xgboost
import joblib
model = joblib.load('model/xgb_model.pkl')

def get_live_features():
    # últimos 100 candles (para calcular médias móveis de 20 e VWAP diário)
    ohlcv_alt = exchange.fetch_ohlcv(par_altcoin, timeframe=timeframe, limit=100)
    ohlcv_btc = exchange.fetch_ohlcv(par_btc, timeframe=timeframe, limit=100)
    
    # dataframes
    cols = ['timestamp', 'open', 'high', 'low', 'close', 'volume']
    df = pd.DataFrame(ohlcv_alt, columns=cols)
    df_btc = pd.DataFrame(ohlcv_btc, columns=cols)
    
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    df_btc['timestamp'] = pd.to_datetime(df_btc['timestamp'], unit='ms')
    
    # merge com BTC
    df_btc = df_btc[['timestamp', 'close']].rename(columns={'close': 'btc_close'})
    df = pd.merge(df, df_btc, on='timestamp', how='left')
    
    ### RECRIAÇÃO DAS FEATURES ###
    # nome e ordem das colunas iguais ao 'features_cols' do treino
    df['log_return'] = np.log(df['close'] / df['close'].shift(1))
    df['btc_log_return'] = np.log(df['btc_close'] / df['btc_close'].shift(1))
    df['forca_relativa_btc'] = df['log_return'] - df['btc_log_return']
    
    df['vol_change'] = (df['volume'] - df['volume'].shift(1)) / df['volume'].shift(1)
    df['momentum'] = (df['close'] - df['close'].shift(3)) / df['close'].shift(3)
    df['hour'] = df['timestamp'].dt.hour
    
    # Z-Score
    ma_20 = df['close'].rolling(window=20).mean()
    std_20 = df['close'].rolling(window=20).std()
    df['z_score_close'] = (df['close'] - ma_20) / std_20
    
    # VWAP Diário
    df['date'] = df['timestamp'].dt.date
    df['typical_price'] = (df['high'] + df['low'] + df['close']) / 3
    df['vp'] = df['typical_price'] * df['volume']
    
    # Soma cumulativa agrupada por dia
    df['cum_vp'] = df.groupby('date')['vp'].cumsum()
    df['cum_vol'] = df.groupby('date')['volume'].cumsum()
    df['daily_VWAP'] = df['cum_vp'] / df['cum_vol']
    df['dist_VWAP'] = (df['close'] - df['daily_VWAP']) / df['daily_VWAP']
    
    # apenas a ÚLTIMA linha (o candle que acabou de fechar) e as colunas do modelo
    features_cols = [
        "log_return"
        # ,"rsi"
        ,"vol_change"
        # ,"z_score_vol" 
        ,"z_score_close"
        # ,"assimetria"
        # ,"curtosi"
        ,"dist_VWAP"
        # , "vol_signal_idx"
        ,"momentum"
        ,"hour"
        ,"btc_log_return"
        ,"forca_relativa_btc"
    ]
    
    latest_features = df.iloc[-1:][features_cols]
    return latest_features, df.iloc[-1]['close']

def run_bot():
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S %Z')}] Par: {par_altcoin} | Timeframe: {timeframe}")
    threshold_otimo = 0.605
    
    while True:
        try:
            # Puxa os dados e calcula as métricas do momento atual
            X_live, preco_atual = get_live_features()
            
            # Pede a probabilidade para o modelo
            prob_alta = model.predict_proba(X_live)[0][1]
            
            print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S %Z')}] Preço: ${preco_atual:.4f} | Probabilidade de alta: {prob_alta*100:.2f}%")
            
            if prob_alta >= threshold_otimo:
                print("==================================================")
                print("SINAL DE COMPRA CONFIRMADO")
                print(f"Gatilho: {prob_alta:.3f} >= {threshold_otimo}")
                print(f"Executando Market Order no preço ~${preco_atual:.4f}")
                # código da exchange, tipo... exchange.create_market_buy_order(...)
                print("==================================================")
            
            # seria bom sincronizar com o relógio para rodar exatamento no minuto 00:01
            time.sleep(60) # 1 hora
            # time.sleep(60 * 60) # 1 hora
            
        except Exception as e:
            print(f"Erro no loop de execução: {e}")
            time.sleep(60) # tenta de novo em 1 minuto se a api falhar

run_bot()